# Project Team Members

| Prepared by | Email | Prepared for |
| :-: | :-: | :-: |
| **_Hardani Ismu Nabil_** | _hardani.ismu.n@gmail.com_ | **_ND Stadium Collaborative Project 2025/2026_** |

# Notebook Target Definition

* **GOAL** create merged app metric dataframes by merge_asof left=df_metric from QualiPoc with right=df_sigcap
* **REQUIRED** to run Ardan-2026_analysis_* notebooks
* **DATE** This is for Postgame February 2026 data

# ===============================
# ===============================

# A-Setup

In [1]:
import os
import pathlib

import numpy as np
import pandas as pd

In [2]:
# ── Path Setup ──────────────────────────────────────────────────────────────
PROJECT_ROOT = pathlib.Path('.').cwd().parent


print('='*50, '[DATA DIRECTORY]')
DATA_DIR3 = PROJECT_ROOT / 'data_raw/2026_all_qp_csv'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Looking for data in: {DATA_DIR3}")


print('='*50, '[DATA DIRECTORY]')
DATA_DIR2 = PROJECT_ROOT / 'data_raw/postseason_sigcap_csv'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Looking for data in: {DATA_DIR2}")

print('='*50, '[DATA DIRECTORY]')
DATA_DIR4 = PROJECT_ROOT / 'data_raw/postseason_qp_csv' #The lab forgot to export messaging, so it is a standalone folder

print(f"Project Root: {PROJECT_ROOT}")
print(f"Looking for data in: {DATA_DIR4}")


print('- '*25)
print('='*50, '[OUTPUT DIRECTORY]')

# OUTPUT_DIR = PROJECT_ROOT / 'output/output_July1st_sigcapDataProcessing/sigcap-merge_postgame'
OUTPUT_DIR = PROJECT_ROOT / 'data_processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"All outputs will be in: {OUTPUT_DIR}")

================================================== [DATA DIRECTORY]
Project Root: /home/analisis/repos/NDKI_01_ForGithub_ForArtifact_wintech_2026_stadium_paper_analysis
Looking for data in: /home/analisis/repos/NDKI_01_ForGithub_ForArtifact_wintech_2026_stadium_paper_analysis/data_raw/2026_all_qp_csv
================================================== [DATA DIRECTORY]
Project Root: /home/analisis/repos/NDKI_01_ForGithub_ForArtifact_wintech_2026_stadium_paper_analysis
Looking for data in: /home/analisis/repos/NDKI_01_ForGithub_ForArtifact_wintech_2026_stadium_paper_analysis/data_raw/postseason_sigcap_csv
================================================== [DATA DIRECTORY]
Project Root: /home/analisis/repos/NDKI_01_ForGithub_ForArtifact_wintech_2026_stadium_paper_analysis
Looking for data in: /home/analisis/repos/NDKI_01_ForGithub_ForArtifact_wintech_2026_stadium_paper_analysis/data_raw/postseason_qp_csv
- - - - - - - - - - - - - - - - - - - - - - - - - 
===================================

# B-SigCap

In [3]:
# ── 1. Load SigCap (Explicit per-file for column reconciliation) ──────────
# T-Mobile
df_tmo = pd.read_csv(DATA_DIR2 / '2026_02_S26_1117_E26_1247Z-0500_outdoor_T-Mobile_a0654bd3_sta_at_tmo_bow.debug_general.csv')
df_vzw = pd.read_csv(DATA_DIR2 / '2026_02_S26_1030_E26_1247Z-0500_outdoor_Verizon_7c36e7df_sta_mr_vzw_bow.debug_general.csv')
df_att = pd.read_csv(DATA_DIR2 / '2026_02_S26_1124_E26_1246Z-0500_outdoor_AT&T_fbd516f0_sta_mr_att_bow.debug_general.csv')


/tmp/ipykernel_29132/3529659859.py:3: DtypeWarning: Columns (0: nr_other1_is_signalStrAPI, 1: wifi_connected_ssid, 2: wifi_connected_bssid, 3: wifi_connected_standard, 4: wifi_2.4_other1_ssid, 5: wifi_2.4_other1_bssid, 6: wifi_2.4_other1_standard, 7: wifi_2.4_other2_ssid, 8: wifi_2.4_other2_bssid, 9: wifi_2.4_other2_standard, 10: wifi_5_other13_ssid, 11: wifi_5_other13_bssid, 12: wifi_5_other13_standard, 13: wifi_5_other14_ssid, 14: wifi_5_other14_bssid, 15: wifi_5_other14_standard, 16: wifi_5_other15_ssid, 17: wifi_5_other15_bssid, 18: wifi_5_other15_standard, 19: wifi_5_other16_ssid, 20: wifi_5_other16_bssid, 21: wifi_5_other16_standard, 22: wifi_5_other17_ssid, 23: wifi_5_other17_bssid, 24: wifi_5_other17_standard, 25: wifi_5_other18_ssid, 26: wifi_5_other18_bssid, 27: wifi_5_other18_standard, 28: wifi_5_other19_ssid, 29: wifi_5_other19_bssid, 30: wifi_5_other19_standard, 31: wifi_5_other20_ssid, 32: wifi_5_other20_bssid, 33: wifi_5_other20_standard, 34: wifi_5_other21_ssid, 35: wif

/tmp/ipykernel_29132/3529659859.py:4: DtypeWarning: Columns (0: wifi_2.4_other1_ssid, 1: wifi_2.4_other1_bssid, 2: wifi_2.4_other1_standard, 3: wifi_5_other126_ssid, 4: wifi_5_other126_bssid, 5: wifi_5_other126_standard, 6: wifi_5_other127_ssid, 7: wifi_5_other127_bssid, 8: wifi_5_other127_standard, 9: wifi_5_other128_ssid, 10: wifi_5_other128_bssid, 11: wifi_5_other128_standard) have mixed types. Specify dtype option on import or set low_memory=False.
  df_vzw = pd.read_csv(DATA_DIR2 / '2026_02_S26_1030_E26_1247Z-0500_outdoor_Verizon_7c36e7df_sta_mr_vzw_bow.debug_general.csv')


In [4]:
for name, df in zip(['TMO', 'VZW', 'ATT'], [df_tmo, df_vzw, df_att]):
    counts = df.columns.value_counts()
    duplicates = counts[counts > 1]
    
    print(f"--- {name} Duplicate Report ---")
    if not duplicates.empty:
        print(f"Found {len(duplicates)} duplicated column names:")
        for col, count in duplicates.items():
            print(f"  - '{col}' appears {count} times")
    else:
        print("No duplicate column names found.")
    print("\n")

--- TMO Duplicate Report ---
No duplicate column names found.


--- VZW Duplicate Report ---
No duplicate column names found.


--- ATT Duplicate Report ---
No duplicate column names found.




In [5]:
import pandas as pd

# 1. Define your target columns and renaming map
selected_cols = [
    'timestamp', 'latitude', 'longitude', 'uuid', 
    'operator', 'Location', 'network_type*', 'wifi_connected_primary_freq_mhz'
]

sigcap_rename = {
    'carrier': 'operator',
    'connected_wifi_primary_freq_mhz': 'wifi_connected_primary_freq_mhz',
    'device_id': 'uuid',
}

# 2. List of dataframes to process
# (Assuming df_tmo, df_vzw, and df_att are already loaded from your first block)
processed_data = []

processed_data = []

for df in [df_tmo, df_vzw, df_att]:
    # 1. Standardize names
    temp = df.rename(columns=sigcap_rename).copy()
    
    # 2. Add/Fix necessary columns
    temp['Location'] = 'Bowl'
    temp['timestamp'] = pd.to_datetime(temp['timestamp'])
    
    if 'uuid' in temp.columns:
        temp['uuid'] = temp['uuid'].astype(str).str.slice(0, 8)

    # --- THE FIX ---
    # We find the index of the first occurrence of each desired column
    # This prevents getting 2 columns when 1 was expected.
    indices = []
    for col in selected_cols:
        loc = temp.columns.get_loc(col)
        # If loc is a boolean array (duplicates), take the first one (True)
        if isinstance(loc, np.ndarray) or isinstance(loc, pd.Index) or not isinstance(loc, int):
            # This handles the duplicate column case by taking the first integer index
            indices.append(np.where(temp.columns == col)[0][0])
        else:
            indices.append(loc)
    
    # Extract using integer positions to guarantee we only get 8 columns
    clean_subset = pd.DataFrame(temp.iloc[:, indices].values, columns=selected_cols)
    # ----------------
    
    # Fix types and append
    clean_subset['timestamp'] = pd.to_datetime(clean_subset['timestamp'])
    processed_data.append(clean_subset)

# 3. Final Concatenation
df_sigcap = pd.concat(processed_data, ignore_index=True)
df_sigcap = df_sigcap.sort_values('timestamp').reset_index(drop=True)

print(f"Sigcap data shape: {df_sigcap.shape}") # Should be (X, 8)
display(df_sigcap.head())

Sigcap data shape: (3056, 8)


,timestamp,latitude,longitude,uuid,operator,Location,network_type*,wifi_connected_primary_freq_mhz
0,2026-02-26 10:30:21.802000-05:00,41.697989,-86.234478,7c36e7df,Verizon,Bowl,LTE,NaN
1,2026-02-26 10:30:26.808000-05:00,41.697994,-86.23447,7c36e7df,Verizon,Bowl,LTE,NaN
2,2026-02-26 10:30:31.811000-05:00,41.698,-86.23447,7c36e7df,Verizon,Bowl,LTE,NaN
3,2026-02-26 10:30:36.815000-05:00,41.698003,-86.234466,7c36e7df,Verizon,Bowl,LTE,NaN
4,2026-02-26 10:30:41.814000-05:00,41.698003,-86.234466,7c36e7df,Verizon,Bowl,LTE,NaN


# ===============================
# ===============================

# C-Tests

### C: Browsing

In [6]:
# 1. Load the single browser file
df_browser = pd.read_csv(DATA_DIR3 / 'browser.csv')
df_browser = df_browser[df_browser['Collection'] == '20260226_preGame_Bowl']

# 2. Assign Location and fix Time
df_browser['Location'] = 'Bowl'
df_browser['Time'] = pd.to_datetime(df_browser['Time'])
df_browser = df_browser.sort_values('Time')

# 3. Map the Units to the 8-char SigCap UUIDs
# FIXED on July 1st
uuid_map = {
    'QualiPoc_619529': 'fbd516f0',   # T-Mobile
    'QualiPoc_361786': 'a0654bd3',   # AT&T
    'QualiPoc_396840': '7c36e7df'    # Verizon
}
print(
    '='*20 + '[SANITY CHECK]' + '='*20, '\n',
    df_sigcap[['uuid', 'operator']].value_counts(), '\n',
    df_browser[['Unit', 'Operator']].value_counts(), '\n',
    '='*20 + '[============]' + '='*20,
)

df_browser['uuid'] = df_browser['Unit'].map(uuid_map)

# 4. Cleanup and Display
print(f"Browser data shape: {df_browser.shape}")
display(df_browser[['Unit', 'Operator', 'uuid']].value_counts())
df_browser.head()

====================[SANITY CHECK]==================== 
 uuid      operator
a0654bd3  T-Mobile    1074
7c36e7df  Verizon      998
fbd516f0  AT&T         984
Name: count, dtype: int64 
 Unit             Operator
QualiPoc_361786  T-Mobile    130
QualiPoc_619529  AT&T        115
QualiPoc_396840  Verizon      90
Name: count, dtype: int64 
 ====================[============]====================
Browser data shape: (335, 35)


Unit             Operator  uuid    
QualiPoc_361786  T-Mobile  a0654bd3    130
QualiPoc_619529  AT&T      fbd516f0    115
QualiPoc_396840  Verizon   7c36e7df     90
Name: count, dtype: int64

,Time,Latitude,Longitude,Collection,Unit,Operator,Technology,Browsing Duration,Download Duration,IP Throughput,...,Host,Protocol,Operation,AP Type,AP Profile Name,Test Name,Direction,Test Status,Location,uuid
0,2026-02-26 11:25:20.590,41.697988,-86.234480,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,1.990,1.491,25705.110001,...,instagram.com,HTTP,GET,default supl mms xcap,T-Mobile US LTE,Browsing,Downlink,Completed,Bowl,a0654bd3
1,2026-02-26 11:25:27.197,41.697982,-86.234494,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,5.475,4.711,7648.774611,...,tiktok.com,HTTP,GET,default supl mms xcap,T-Mobile US LTE,Browsing,Downlink,Completed,Bowl,a0654bd3
2,2026-02-26 11:25:30.708,41.697987,-86.234502,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,2.368,1.954,5617.594815,...,facebook.com,HTTP,GET,default supl mms xcap,T-Mobile US LTE,Browsing,Downlink,Completed,Bowl,a0654bd3
3,2026-02-26 11:25:36.681,41.697987,-86.234511,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,4.837,4.217,22068.077615,...,ncaa.com,HTTP,GET,default supl mms xcap,T-Mobile US LTE,Browsing,Downlink,Completed,Bowl,a0654bd3
4,2026-02-26 11:25:43.327,41.697986,-86.234515,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,5.494,5.136,6367.353761,...,sports.yahoo.com,HTTP,GET,default supl mms xcap,T-Mobile US LTE,Browsing,Downlink,Completed,Bowl,a0654bd3


In [7]:
print(
    df_sigcap[['uuid', 'operator']].value_counts(),
    df_browser[['Unit', 'Operator']].value_counts()
)

uuid      operator
a0654bd3  T-Mobile    1074
7c36e7df  Verizon      998
fbd516f0  AT&T         984
Name: count, dtype: int64 Unit             Operator
QualiPoc_361786  T-Mobile    130
QualiPoc_619529  AT&T        115
QualiPoc_396840  Verizon      90
Name: count, dtype: int64


In [8]:
import numpy as np

# 1. Handle Timezones
# Localize Browser time to ET and convert to UTC
if df_browser['Time'].dt.tz is None:
    df_browser['Time'] = df_browser['Time'].dt.tz_localize('US/Eastern', ambiguous='infer')

# Ensure Sigcap is UTC-ready (assuming it's already aware or UTC based on your first block)
df_browser['Time_utc'] = df_browser['Time'].dt.tz_convert('UTC')
df_sigcap['timestamp_utc'] = df_sigcap['timestamp'].dt.tz_convert('UTC')

# 2. Merge
# We sort both here to ensure merge_asof requirements are met
cols_to_merge = ['timestamp_utc', 'uuid', 'operator', 'network_type*', 'wifi_connected_primary_freq_mhz', 'Location']

df_browser_merged = pd.merge_asof(
    df_browser.sort_values('Time_utc'),
    df_sigcap[cols_to_merge].sort_values('timestamp_utc'),
    left_on='Time_utc',
    right_on='timestamp_utc',
    by='uuid',
    direction='nearest',
    tolerance=pd.Timedelta('10s')
)

# 3. Handle Operator Overwriting & preservation
# Preserve the original carrier name (e.g., T-Mobile) before we overwrite it
df_browser_merged['carrier_original'] = df_browser_merged['operator']

conditions = [
    (df_browser_merged['wifi_connected_primary_freq_mhz'].between(5000, 5925, inclusive='both')),
    (df_browser_merged['wifi_connected_primary_freq_mhz'] > 5925),
    (df_browser_merged['wifi_connected_primary_freq_mhz'] < 5000)
]
choices = ['Wi-Fi 5', 'Wi-Fi 6', 'Wi-Fi 2.4']

# Overwrite 'operator' with Wi-Fi info where applicable, otherwise keep the carrier name
df_browser_merged['operator'] = np.select(conditions, choices, default=df_browser_merged['operator'])

# 4. Cleanup
df_browser_merged = df_browser_merged.drop(columns=['timestamp_utc', 'Time_utc'])
df_browser_merged = df_browser_merged.rename(columns={'Location_x': 'Location', 'Location_y': 'Location_sigcap'})

# Final check
print(f"Merged shape: {df_browser_merged.shape}")
display(df_browser_merged[['operator', 'carrier_original', 'wifi_connected_primary_freq_mhz']].head(10))

# df_browser_merged.to_csv('../data/2026NdSta/df_browser_merged.csv', index=False)
df_browser_merged.to_csv(OUTPUT_DIR / 'df_browser_merged_pg.csv', index=False)

Merged shape: (335, 40)


,operator,carrier_original,wifi_connected_primary_freq_mhz
0,Wi-Fi 6,T-Mobile,6375.0
1,Wi-Fi 6,T-Mobile,6375.0
2,Wi-Fi 6,T-Mobile,6375.0
3,Wi-Fi 6,T-Mobile,6375.0
4,Wi-Fi 6,T-Mobile,6375.0
5,AT&T,AT&T,NaN
6,AT&T,AT&T,NaN
7,AT&T,AT&T,NaN
8,AT&T,AT&T,NaN
9,AT&T,AT&T,NaN


In [9]:
df_browser_merged['operator'].value_counts()

operator
Wi-Fi 6     125
AT&T         80
T-Mobile     65
Verizon      60
Wi-Fi 5       5
Name: count, dtype: int64

### C: Messaging

In [10]:
# df_messaging.groupby(['Operator'])['Time'].describe()

In [11]:
# 1. Load the single browser file
df_messaging = pd.read_csv(DATA_DIR4 / 'app_messaging_postgame.csv')
df_messaging = df_messaging[df_messaging['Collection'] == '20260226_preGame_Bowl']

# 2. Assign Location and fix Time
df_messaging['Location'] = 'Bowl'
df_messaging['Time'] = pd.to_datetime(df_messaging['Time'])
df_messaging = df_messaging.sort_values('Time')

# 3. Map the Units to the 8-char SigCap UUIDs
# FIXED on July 1st
uuid_map = {
    'QualiPoc_619529': 'fbd516f0',   # T-Mobile
    'QualiPoc_361786': 'a0654bd3',   # AT&T
    'QualiPoc_396840': '7c36e7df'    # Verizon
}
print(
    '='*20 + '[SANITY CHECK]' + '='*20, '\n',
    df_sigcap[['uuid', 'operator']].value_counts(), '\n',
    df_messaging[['Unit', 'Operator']].value_counts(), '\n',
    '='*20 + '[============]' + '='*20,
)

df_messaging['uuid'] = df_messaging['Unit'].map(uuid_map)

# df_messaging = df_messaging[df_messaging['Time'] < '2026-02-27 00:00:00']

# 4. Cleanup and Display
print(f"Browser data shape: {df_messaging.shape}")
display(df_messaging[['Unit', 'Operator', 'uuid']].value_counts())
df_messaging.head()

====================[SANITY CHECK]==================== 
 uuid      operator
a0654bd3  T-Mobile    1074
7c36e7df  Verizon      998
fbd516f0  AT&T         984
Name: count, dtype: int64 
 Unit             Operator
QualiPoc_361786  T-Mobile    52
QualiPoc_619529  AT&T        42
QualiPoc_396840  Verizon     36
Name: count, dtype: int64 
 ====================[============]====================
Browser data shape: (130, 20)


Unit             Operator  uuid    
QualiPoc_361786  T-Mobile  a0654bd3    52
QualiPoc_619529  AT&T      fbd516f0    42
QualiPoc_396840  Verizon   7c36e7df    36
Name: count, dtype: int64

,Time,Latitude,Longitude,Collection,Unit,Operator,Technology,Send Duration,Delivery Duration,Start Time,Receive Time,Transmission Status,Delivery Status,Delivery Notification,Action Type,File Size,Measurement Name,Test Status,Location,uuid
0,2026-02-26 11:25:58.255,41.697984,-86.234509,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,0.354,1.346,2026-02-26 11:25:56.110,NaN,Successful,Successful,Enabled,Send Text,NaN,WhatsApp Send,Completed,Bowl,a0654bd3
1,2026-02-26 11:26:08.882,41.697981,-86.234498,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,1.491,2.423,2026-02-26 11:26:05.789,NaN,Successful,Successful,Enabled,Send Picture,999215.0,WhatsApp Send,Completed,Bowl,a0654bd3
2,2026-02-26 11:27:16.725,41.698059,-86.234443,20260226_preGame_Bowl,QualiPoc_619529,AT&T,LTE,0.303,2.211,2026-02-26 11:27:14.247,NaN,Successful,Successful,Enabled,Send Text,NaN,WhatsApp Send,Completed,Bowl,fbd516f0
3,2026-02-26 11:27:52.932,41.698061,-86.234444,20260226_preGame_Bowl,QualiPoc_619529,AT&T,LTE,NaN,NaN,2026-02-26 11:27:23.245,NaN,Failed,Failed,Enabled,Send Picture,999493.0,WhatsApp Send,Failed,Bowl,fbd516f0
4,2026-02-26 11:28:14.755,41.697994,-86.234465,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,0.300,0.701,2026-02-26 11:28:13.315,NaN,Successful,Successful,Enabled,Send Text,NaN,WhatsApp Send,Completed,Bowl,a0654bd3


In [12]:
import numpy as np

# 1. Handle Timezones
# Localize Browser time to ET and convert to UTC
if df_messaging['Time'].dt.tz is None:
    df_messaging['Time'] = df_messaging['Time'].dt.tz_localize('US/Eastern', ambiguous='infer')

# Ensure Sigcap is UTC-ready (assuming it's already aware or UTC based on your first block)
df_messaging['Time_utc'] = df_messaging['Time'].dt.tz_convert('UTC')
df_sigcap['timestamp_utc'] = df_sigcap['timestamp'].dt.tz_convert('UTC')

# 2. Merge
# We sort both here to ensure merge_asof requirements are met
cols_to_merge = ['timestamp_utc', 'uuid', 'operator', 'network_type*', 'wifi_connected_primary_freq_mhz', 'Location']

df_messaging_merged = pd.merge_asof(
    df_messaging.sort_values('Time_utc'),
    df_sigcap[cols_to_merge].sort_values('timestamp_utc'),
    left_on='Time_utc',
    right_on='timestamp_utc',
    by='uuid',
    direction='nearest',
    tolerance=pd.Timedelta('10s')
)

# 3. Handle Operator Overwriting & preservation
# Preserve the original carrier name (e.g., T-Mobile) before we overwrite it
df_messaging_merged['carrier_original'] = df_messaging_merged['operator']

conditions = [
    (df_messaging_merged['wifi_connected_primary_freq_mhz'].between(5000, 5925, inclusive='both')),
    (df_messaging_merged['wifi_connected_primary_freq_mhz'] > 5925),
    (df_messaging_merged['wifi_connected_primary_freq_mhz'] < 5000)
]
choices = ['Wi-Fi 5', 'Wi-Fi 6', 'Wi-Fi 2.4']

# Overwrite 'operator' with Wi-Fi info where applicable, otherwise keep the carrier name
df_messaging_merged['operator'] = np.select(conditions, choices, default=df_messaging_merged['operator'])

# 4. Cleanup
df_messaging_merged = df_messaging_merged.drop(columns=['timestamp_utc', 'Time_utc'])
df_messaging_merged = df_messaging_merged.rename(columns={'Location_x': 'Location', 'Location_y': 'Location_sigcap'})

# Final check
print(f"Merged shape: {df_messaging_merged.shape}")
display(df_messaging_merged[['operator', 'carrier_original', 'wifi_connected_primary_freq_mhz']].head(10))

# df_messaging_merged.to_csv('../data/2026NdSta/df_messaging_merged.csv', index=False)
df_messaging_merged.to_csv(OUTPUT_DIR / 'df_messaging_merged_pg.csv', index=False)

Merged shape: (130, 25)


,operator,carrier_original,wifi_connected_primary_freq_mhz
0,Wi-Fi 6,T-Mobile,6375.0
1,Wi-Fi 6,T-Mobile,6375.0
2,AT&T,AT&T,NaN
3,AT&T,AT&T,NaN
4,T-Mobile,T-Mobile,NaN
5,T-Mobile,T-Mobile,NaN
6,T-Mobile,T-Mobile,NaN
7,T-Mobile,T-Mobile,NaN
8,AT&T,AT&T,NaN
9,AT&T,AT&T,NaN


### C: Instagram

In [13]:
# 1. Load the single browser file
df_instagram = pd.read_csv(DATA_DIR3 / 'data_app.csv')
df_instagram = df_instagram[df_instagram['Collection'] == '20260226_preGame_Bowl']

# 2. Assign Location and fix Time
df_instagram['Location'] = 'Bowl'
df_instagram['Time'] = pd.to_datetime(df_instagram['Time'])
df_instagram = df_instagram.sort_values('Time')

df_instagram = df_instagram[df_instagram['Test Name'] == 'Instagram']

# 3. Map the Units to the 8-char SigCap UUIDs
# FIXED on July 1st
uuid_map = {
    'QualiPoc_619529': 'fbd516f0',   # T-Mobile
    'QualiPoc_361786': 'a0654bd3',   # AT&T
    'QualiPoc_396840': '7c36e7df'    # Verizon
}
print(
    '='*20 + '[SANITY CHECK]' + '='*20, '\n',
    df_sigcap[['uuid', 'operator']].value_counts(), '\n',
    df_instagram[['Unit', 'Operator']].value_counts(), '\n',
    '='*20 + '[============]' + '='*20,
)

df_instagram['uuid'] = df_instagram['Unit'].map(uuid_map)

# 4. Cleanup and Display
print(f"Browser data shape: {df_instagram.shape}")
display(df_instagram[['Unit', 'Operator', 'uuid']].value_counts())
df_instagram.head()

====================[SANITY CHECK]==================== 
 uuid      operator
a0654bd3  T-Mobile    1074
7c36e7df  Verizon      998
fbd516f0  AT&T         984
Name: count, dtype: int64 
 Unit             Operator
QualiPoc_361786  T-Mobile    26
QualiPoc_619529  AT&T        21
QualiPoc_396840  Verizon     18
Name: count, dtype: int64 
 ====================[============]====================
Browser data shape: (65, 27)


Unit             Operator  uuid    
QualiPoc_361786  T-Mobile  a0654bd3    26
QualiPoc_619529  AT&T      fbd516f0    21
QualiPoc_396840  Verizon   7c36e7df    18
Name: count, dtype: int64

,Time,Latitude,Longitude,Collection,Unit,Operator,Technology,Data Test,Data Test Action,Direction,...,Max IP Throughput DL,IP Throughput UL,Max IP Throughput UL,Bytes Transferred DL,Bytes Transferred UL,Test Name,Direction.1,Test Status,Location,uuid
5,2026-02-26 11:25:54.555,41.697989,-86.234507,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,Instagram,Post Picture,SEND,...,63532.776,2059.336,13774.824,NaN,999321.0,Instagram,Send,Completed,Bowl,a0654bd3
15,2026-02-26 11:27:12.787,41.698059,-86.234443,20260226_preGame_Bowl,QualiPoc_619529,AT&T,LTE,Instagram,Post Picture,SEND,...,3889.832,792.384,7985.296,NaN,999175.0,Instagram,Send,Completed,Bowl,fbd516f0
22,2026-02-26 11:28:11.667,41.697994,-86.234466,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,Instagram,Post Picture,SEND,...,24137.944,1153.824,7162.856,NaN,999215.0,Instagram,Send,Completed,Bowl,a0654bd3
36,2026-02-26 11:31:18.640,41.697989,-86.234690,20260226_preGame_Bowl,QualiPoc_361786,T-Mobile,5G NR,Instagram,Post Picture,SEND,...,28836.792,1512.816,11575.544,NaN,998969.0,Instagram,Send,Completed,Bowl,a0654bd3
38,2026-02-26 11:31:34.281,41.697977,-86.234462,20260226_preGame_Bowl,QualiPoc_619529,AT&T,LTE,Instagram,Post Picture,SEND,...,4671.640,1034.488,8312.040,NaN,999493.0,Instagram,Send,Completed,Bowl,fbd516f0


In [14]:
import numpy as np

# 1. Handle Timezones
# Localize Browser time to ET and convert to UTC
if df_instagram['Time'].dt.tz is None:
    df_instagram['Time'] = df_instagram['Time'].dt.tz_localize('US/Eastern', ambiguous='infer')

# Ensure Sigcap is UTC-ready (assuming it's already aware or UTC based on your first block)
df_instagram['Time_utc'] = df_instagram['Time'].dt.tz_convert('UTC')
df_sigcap['timestamp_utc'] = df_sigcap['timestamp'].dt.tz_convert('UTC')

# 2. Merge
# We sort both here to ensure merge_asof requirements are met
cols_to_merge = ['timestamp_utc', 'uuid', 'operator', 'network_type*', 'wifi_connected_primary_freq_mhz', 'Location']

df_instagram_merged = pd.merge_asof(
    df_instagram.sort_values('Time_utc'),
    df_sigcap[cols_to_merge].sort_values('timestamp_utc'),
    left_on='Time_utc',
    right_on='timestamp_utc',
    by='uuid',
    direction='nearest',
    tolerance=pd.Timedelta('10s')
)

# 3. Handle Operator Overwriting & preservation
# Preserve the original carrier name (e.g., T-Mobile) before we overwrite it
df_instagram_merged['carrier_original'] = df_instagram_merged['operator']

conditions = [
    (df_instagram_merged['wifi_connected_primary_freq_mhz'].between(5000, 5925, inclusive='both')),
    (df_instagram_merged['wifi_connected_primary_freq_mhz'] > 5925),
    (df_instagram_merged['wifi_connected_primary_freq_mhz'] < 5000)
]
choices = ['Wi-Fi 5', 'Wi-Fi 6', 'Wi-Fi 2.4']

# Overwrite 'operator' with Wi-Fi info where applicable, otherwise keep the carrier name
df_instagram_merged['operator'] = np.select(conditions, choices, default=df_instagram_merged['operator'])

# 4. Cleanup
df_instagram_merged = df_instagram_merged.drop(columns=['timestamp_utc', 'Time_utc'])
df_instagram_merged = df_instagram_merged.rename(columns={'Location_x': 'Location', 'Location_y': 'Location_sigcap'})

# Final check
print(f"Merged shape: {df_instagram_merged.shape}")
display(df_instagram_merged[['operator', 'carrier_original', 'wifi_connected_primary_freq_mhz']].head(10))

# df_instagram_merged.to_csv('../data/2026NdSta/df_instagram_merged.csv', index=False)
df_instagram_merged.to_csv(OUTPUT_DIR / 'df_instagram_merged_pg.csv', index=False)

Merged shape: (65, 32)


,operator,carrier_original,wifi_connected_primary_freq_mhz
0,Wi-Fi 6,T-Mobile,6375.0
1,AT&T,AT&T,NaN
2,T-Mobile,T-Mobile,NaN
3,T-Mobile,T-Mobile,NaN
4,AT&T,AT&T,NaN
5,Wi-Fi 6,T-Mobile,6355.0
6,Wi-Fi 6,AT&T,5975.0
7,Wi-Fi 6,Verizon,6375.0
8,Wi-Fi 6,T-Mobile,6355.0
9,AT&T,AT&T,NaN
